<a href="https://colab.research.google.com/github/alvesnicole/Fato-ou-Fake-IA/blob/main/comparacao_modelos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ── 0. Instala dependências ──────────────────────────────────────────────────
!pip install -q pandas scikit-learn matplotlib seaborn

# ── 1. Upload das planilhas ──────────────────────────────────────────────────
from google.colab import files
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (classification_report, confusion_matrix,
                             accuracy_score, f1_score, precision_score, recall_score)

print("Envie a planilha do QWEN:")
up1 = files.upload()
print("Envie a planilha do LLAMA:")
up2 = files.upload()

df_qwen  = pd.read_csv(list(up1.keys())[0])
df_llama = pd.read_csv(list(up2.keys())[0])

# ── 2. Padroniza nomes de coluna ─────────────────────────────────────────────
COL_QWEN  = [c for c in df_qwen.columns  if c.startswith("pred_")][0]
COL_LLAMA = [c for c in df_llama.columns if c.startswith("pred_")][0]
LAT_QWEN  = [c for c in df_qwen.columns  if c.startswith("lat_")][0]
LAT_LLAMA = [c for c in df_llama.columns if c.startswith("lat_")][0]

df_qwen  = df_qwen.rename(columns={COL_QWEN:  "pred", LAT_QWEN:  "lat"})
df_llama = df_llama.rename(columns={COL_LLAMA: "pred", LAT_LLAMA: "lat"})

# Remove erros
df_qwen  = df_qwen[df_qwen["pred"]  != "ERRO"].copy()
df_llama = df_llama[df_llama["pred"] != "ERRO"].copy()

# ── 3. Métricas individuais ──────────────────────────────────────────────────
def metricas(df, nome):
    y_true = df["label_real"]
    y_pred = df["pred"]
    return {
        "Modelo":     nome,
        "Acurácia":   round(accuracy_score(y_true, y_pred), 4),
        "Precisão":   round(precision_score(y_true, y_pred, pos_label="FALSO"), 4),  # ← trocado
        "Recall":     round(recall_score(y_true, y_pred,    pos_label="FALSO"), 4),  # ← trocado
        "F1-Score":   round(f1_score(y_true, y_pred,        pos_label="FALSO"), 4),  # ← trocado
        "Latência média (s)": round(df["lat"].mean(), 4),
        "Total notícias": len(df),
    }

m_qwen  = metricas(df_qwen,  "Qwen3-32b")
m_llama = metricas(df_llama, "Llama-3.3-70b")

df_comp = pd.DataFrame([m_qwen, m_llama]).set_index("Modelo")
print("\n=== Tabela Comparativa ===")
print(df_comp.to_string())

# ── 4. Relatório detalhado ───────────────────────────────────────────────────
for nome, df in [("Qwen3-32b", df_qwen), ("Llama-3.3-70b", df_llama)]:
    print(f"\n--- {nome} ---")
    print(classification_report(df["label_real"], df["pred"],
                                 target_names=["FALSO", "VERDADEIRO"]))

# ── 5. Gráficos ──────────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle("Comparação de Modelos — Detecção de Fake News", fontsize=15, fontweight="bold")

# 5.1 Matrizes de confusão
for ax, (nome, df) in zip(axes[0, :2], [("Qwen3-32b", df_qwen), ("Llama-3.3-70b", df_llama)]):
    cm = confusion_matrix(df["label_real"], df["pred"], labels=["FALSO", "VERDADEIRO"])
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax,
                xticklabels=["FALSO", "VERDADEIRO"],
                yticklabels=["FALSO", "VERDADEIRO"])
    ax.set_title(f"Matriz de Confusão\n{nome}")
    ax.set_ylabel("Real")
    ax.set_xlabel("Predito")

# 5.2 Barras — métricas comparativas
metricas_cols = ["Acurácia", "Precisão", "Recall", "F1-Score"]
x = np.arange(len(metricas_cols))
w = 0.35
ax = axes[0, 2]
ax.bar(x - w/2, [m_qwen[c]  for c in metricas_cols], w, label="Qwen3-32b",    color="#4C72B0")
ax.bar(x + w/2, [m_llama[c] for c in metricas_cols], w, label="Llama-3.3-70b", color="#DD8452")
ax.set_xticks(x)
ax.set_xticklabels(metricas_cols)
ax.set_ylim(0, 1.1)
ax.set_title("Métricas Comparativas")
ax.legend()
ax.set_ylabel("Score")
for bar in ax.patches:
    ax.annotate(f"{bar.get_height():.2f}",
                (bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01),
                ha="center", fontsize=8)

# 5.3 Distribuição de predições
for ax, (nome, df) in zip(axes[1, :2], [("Qwen3-32b", df_qwen), ("Llama-3.3-70b", df_llama)]):
    contagem = df["pred"].value_counts()
    ax.pie(contagem, labels=contagem.index, autopct="%1.1f%%",
           colors=["#DD8452", "#4C72B0"])
    ax.set_title(f"Distribuição de Predições\n{nome}")

# 5.4 Latência
ax = axes[1, 2]
nomes   = ["Qwen3-32b", "Llama-3.3-70b"]
latencias = [m_qwen["Latência média (s)"], m_llama["Latência média (s)"]]
bars = ax.bar(nomes, latencias, color=["#4C72B0", "#DD8452"])
ax.set_title("Latência Média por Requisição")
ax.set_ylabel("Segundos")
for bar, val in zip(bars, latencias):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f"{val:.2f}s", ha="center", fontweight="bold")

plt.tight_layout()
plt.savefig("comparacao_modelos.png", dpi=150, bbox_inches="tight")
plt.show()
print("\nGráfico salvo como comparacao_modelos.png")

Envie a planilha do QWEN:


Saving final.csv to final (3).csv
Envie a planilha do LLAMA:


Saving tabela_final_llama_versatile.csv to tabela_final_llama_versatile (4).csv

=== Tabela Comparativa ===
               Acurácia  Precisão  Recall  F1-Score  Latência média (s)  Total notícias
Modelo                                                                                 
Qwen3-32b        0.7343    0.9256  0.5051    0.6535              1.4716             794
Llama-3.3-70b    0.7538    0.9511  0.5350    0.6848              0.2783             800

--- Qwen3-32b ---
              precision    recall  f1-score   support

       FALSO       0.93      0.51      0.65       394
  VERDADEIRO       0.66      0.96      0.78       400

    accuracy                           0.73       794
   macro avg       0.79      0.73      0.72       794
weighted avg       0.79      0.73      0.72       794


--- Llama-3.3-70b ---
              precision    recall  f1-score   support

       FALSO       0.95      0.54      0.68       400
  VERDADEIRO       0.68      0.97      0.80       400

    acc